# Figure 1 $p_{ij}$ example
This notebook plots the barcharts to explain how the chance to connect soource node $i$ to target node $j$, $p_{ij}$, depends on the chosen link formation mechanisms and available target nodes.

In [ ]:
from typing import Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import networkx as nx

from netin.link_formation_mechanisms import (
    Uniform, PreferentialAttachment,
    TwoClassHomophily, TriadicClosure)
from netin.graphs import Graph, BinaryClassNodeVector, NodeVector
from netin.models import PATCHModel
from netin.utils import CLASS_ATTRIBUTE, MINORITY_VALUE

from patch.constants import COLOR_MAJ, COLOR_MIN, PATH_PLOTS, SIZE_FIG, MAP_LFM_SHORT
from patch.model_config import ModelConfig, CompoundLFM
from patch.statistics import compute_ei, compute_gini, compute_mann_whitney, compute_gini_min, compute_gini_maj

In [ ]:
# Custom network (see Figure 1)
EDGES = [
    (0,1), (0,2), (0,5), (0,7), (0,9), (1,2),
    (1,4), (2,3), (2,4), (5,6), (5,7), (5,8),
    (6,7), (6,8), (7,8)]

H = 0.8 # Custom homophily parameter

# Minority nodes
MIN = BinaryClassNodeVector.from_ndarray(np.asarray([0,1,1,1,1,0,0,0,0,0]))

# Target nodes (all)
TARGET_SELECTION = list(range(1, len(MIN) - 1))

TARGET_LABELS = list(range(1, len(MIN) - 1))

# Custom figure size
SIZE_FIG_SMALL=(1.065, 1.73)

In [ ]:
plt.rcParams['figure.figsize'] = SIZE_FIG_SMALL
plt.rcParams['font.size'] = 8
plt.rcParams['figure.constrained_layout.use'] = True

## Example network

Create a custom network.

In [ ]:
g = Graph()

for node in range(len(MIN)):
    g.add_node(node)

for edge in EDGES:
    g.add_edge(*edge)

Define link formation mechanisms to compute $p_{ij}$.

In [ ]:
h = TwoClassHomophily(homophily=H, node_class_values=MIN)
pa = PreferentialAttachment(N=len(MIN), graph=g)
u = Uniform(N=len(MIN))
tc = TriadicClosure(N=len(MIN), graph=g)

In [ ]:
p_u = u.get_target_probabilities(source=len(MIN) - 1)
p_h = h.get_target_probabilities(source=len(MIN) - 1)
p_pa = pa.get_target_probabilities(source=len(MIN) - 1)

p_pah = p_pa * p_h
p_pah /= np.sum(p_pah)

p_tc = tc.get_target_probabilities(source=len(MIN) - 1)

## Plotting

In [ ]:
def plot_pij(
        pij: NodeVector,
        lfm: str,
        color_groups: bool = False,
        tpl_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    """Plots a single `p_ij` bar chart.

    Parameters
    ----------
    pij : NodeVector
        The target probabilities $p_{ij}$ for the source node $i$.
    lfm : str
        The link formation mechanism used to compute the probabilities.
    color_groups : bool, optional
        Whether to color the bars by group, by default False
    tpl_ax : tuple, optional
        The matplotlib axes to plot on, by default None

    Returns
    -------
    fig, ax : tuple
        The figure and axes objects of the plot.
    """
    fig_exists = tpl_ax is not None
    fig, ax = plt.subplots(figsize=SIZE_FIG_SMALL) if not fig_exists else tpl_ax
    ax.bar(
        range(len(TARGET_SELECTION)),
        pij[TARGET_SELECTION] / pij[TARGET_SELECTION].sum(),
        color=[COLOR_MIN if MIN[i] else COLOR_MAJ for i in TARGET_SELECTION] if color_groups else "black",
        label='Uniform'
    )

    if not fig_exists:
        ax.set_ylabel(f'$p^{{{lfm}}}_{{ij}}$', labelpad=0)
        ax.set_xlabel('target $j$', labelpad=0)
        # Set xticks and reduce spacing
        ax.set_xticks(
            ticks=range(len(TARGET_SELECTION)),
            labels=[f"${s}$" for s in TARGET_LABELS],
            # Align text baseline (not including descenders like g, y, p)
            va="baseline",
            )
        # Add more spacing between axis and tick labels to prevent overlap
        ax.tick_params(axis='x', pad=8)

    # Remove spines
    ax.spines[['top', 'right']].set_visible(False)
    # Remove y axis ticks
    ax.yaxis.set_ticks([])
    ax.set_ylim(0,.3)

    if not fig_exists:
        fig.tight_layout()
    return fig, ax


In [ ]:
f = plt.figure(
    figsize=(2*SIZE_FIG_SMALL[0], SIZE_FIG_SMALL[1]),
    layout="constrained")
grid = plt.GridSpec(
    nrows=4, ncols=3, figure=f)

ax_tc_u = f.add_subplot(grid[1, 0])
ax_tc_h = f.add_subplot(grid[2, 0],
    sharey=ax_tc_u)
ax_tc_pah = f.add_subplot(grid[3, 0],
    sharey=ax_tc_u)

ax_g_h = f.add_subplot(grid[0, 1],
    sharey=ax_tc_u)
ax_g_pah = f.add_subplot(grid[0, 2],
    sharey=ax_tc_u)

ax_v_h_u = f.add_subplot(grid[1, 1])
ax_v_pah_u = f.add_subplot(grid[1, 2])
ax_v_h_h = f.add_subplot(grid[2, 1])
ax_v_pah_pah = f.add_subplot(grid[3, 2])

l_axes_hist = [ax_g_pah, ax_tc_pah, ax_g_h, ax_tc_h, ax_tc_u]
l_axes_var = [ax_v_pah_u, ax_v_h_u, ax_v_pah_pah, ax_v_h_h]

for p, tc, label, color, ax in zip(
    (p_pah, p_pah, p_h, p_h, p_u),
    [False, True, False, True, True],
    ["PAH", "PAH", "H", "H", "U"],
    [True, True, True, True, False],
    l_axes_hist
):
    p = p * p_tc if tc else p
    p = p / p.sum()  # Normalize probabilities
    plot_pij(
        pij=p,
        lfm=label,
        color_groups=color,
        tpl_ax=(f, ax))
    ax.text(
        x=0.05,
        y=1.,
        s=label,
        ha="left",
        va="top",
        fontsize=8,
        transform=ax.transAxes,
    )

for ax, label in zip(
    l_axes_var,
    ("PAH,U", "H,U", "PAH,PAH", "H,H")):
    ax.text(
        x=.5,
        y=.5,
        s=label,
        ha="center",
        va="center",
        fontsize=8,
        fontweight="bold",
        transform=ax.transAxes,)
    # Hide axes
    ax.axis('off')

for ax in l_axes_hist:
    ax.set_ylim(0, .45)

# Set xticks and reduce spacing
ax_tc_pah.set_xlabel('target $j$', labelpad=0)
ax_tc_pah.set_xticks(
    ticks=range(len(TARGET_SELECTION)),
    labels=[f"${s}$" for s in TARGET_LABELS],
    # Align text baseline (not including descenders like g, y, p)
    va="baseline",)

ax_tc_pah.tick_params(axis='x', pad=7)

for ax in l_axes_hist:
    if ax == ax_tc_pah:
        continue
    ax.set_xticks(range(len(TARGET_SELECTION)), labels=[])

for ax in [ax_tc_u, ax_tc_h, ax_tc_pah]:
    ax.set_ylabel('$p_{ij}$', labelpad=-.050)

f.text(
    x=.33, y=.9, s="$p_{ij}$",
    ha="left", va="baseline", fontsize=8, rotation=90
)
f.text(
    ha="left", va="baseline", fontsize=8,
    x=0.6, y=1.025, s="global",
)
f.text(
    x=-0.025, y=0.35, s="triadic closure",
    ha="left", va="baseline", fontsize=8, rotation=90
)
f.savefig(f"../{PATH_PLOTS}/pij_ALL_var.pdf")

## Sample networks

In [ ]:
# seed = 2
seed = 5

In [ ]:
model_configs = [
    ModelConfig(
        N=30, m=3, f_m=.25,
        realization=0,
        **config
    ) for config in (
        {"homophily": 0.99, "tau": 0.75, "lfm_global": CompoundLFM.HOMOPHILY, "lfm_tc":CompoundLFM.HOMOPHILY},
        {"homophily": 0.01, "tau": 0.0, "lfm_global": CompoundLFM.PAH, "lfm_tc":CompoundLFM.UNIFORM},
        {"homophily": 0.75, "tau": 0.75, "lfm_global": CompoundLFM.PAH, "lfm_tc":CompoundLFM.PAH},
    )
]
graphs, degrees, minorities = [], [], []
for model_config in model_configs:
    model = PATCHModel(**model_config.to_dict(split_homophily=True), seed=seed)
    graph = model.simulate()

    print((
        f"Stats for {model_config}"
        f"\n\tEI: {compute_ei(graph)}"
        f"\n\tGini: {compute_gini(graph.degrees())}"
        f"\n\tGini_min: {compute_gini_min(graph)}"
        f"\n\tGini_maj: {compute_gini_maj(graph)}"
        f"\n\tMW: {compute_mann_whitney(graph)}"
    ))

    degrees.append(graph.degrees())
    minorities.append(graph.get_node_class(CLASS_ATTRIBUTE))
    graphs.append(graph.to_nxgraph())


In [ ]:
positions = [
    nx.spring_layout(graphs[0], seed=seed, pos={
        n: (0.5 + np.random.random(), -.5  + np.random.random())
            if graphs[0].nodes[n][CLASS_ATTRIBUTE] == MINORITY_VALUE else\
                (-.5 + np.random.random(), 0.5 + np.random.random())\
                    for n in graphs[0].nodes},
        iterations=500),
    nx.spring_layout(graphs[1], seed=seed, pos={
        n: (0.5 + np.random.random(), -.5  + np.random.random())
            if graphs[1].nodes[n][CLASS_ATTRIBUTE] == MINORITY_VALUE else\
                (-.5 + np.random.random(), 0.5 + np.random.random())\
                    for n in graphs[1].nodes},
        iterations=500),
    nx.spring_layout(graphs[2], seed=seed, pos={
        n: (0.5 + np.random.random(), -.5  + np.random.random())
            if graphs[2].nodes[n][CLASS_ATTRIBUTE] == MINORITY_VALUE else\
                (-.5 + np.random.random(), 0.5 + np.random.random())\
                    for n in graphs[2].nodes},
        iterations=500),
    ]

In [ ]:
def plot_lorenz_curve(
        degrees: NodeVector,
        minority: BinaryClassNodeVector,
        ax: plt.Axes
):
    for is_minority, color in zip((True, False), (COLOR_MIN, COLOR_MAJ)):
        degrees_group = degrees[minority==is_minority]
        sorted_degrees = np.sort(degrees_group)
        n = len(degrees_group)
        cum_degrees = np.cumsum(sorted_degrees)
        total_degree = cum_degrees[-1]
        lorenz_curve = np.insert(cum_degrees / total_degree, 0, 0)
        x = np.linspace(0, 1, n + 1)
        ax.plot(x, lorenz_curve, drawstyle='steps-post', color=color)
    ax.plot([0, 1], [0, 1], linestyle=':', color='gray')
    # ax.fill_between(x, lorenz_curve, x, step='post', alpha=0.2, color='black')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(
        0.5, .95, 'Lorenz',
        fontsize=8,
        ha="center", va="top",
        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            boxstyle='round,pad=0.1'
        ),
        transform=ax.transAxes)


In [ ]:
def plot_ccdf(
        degrees: NodeVector,
        minority: BinaryClassNodeVector,
        ax: plt.Axes
):
    for is_minority, color in zip((True, False), (COLOR_MIN, COLOR_MAJ)):
        degrees_group = degrees[minority==is_minority]
        unique, counts = np.unique(degrees_group, return_counts=True)
        sorted_indices = np.argsort(unique)
        unique = unique[sorted_indices]
        counts = counts[sorted_indices]
        ccdf = 1 - np.cumsum(counts) / counts.sum()
        ax.step(unique, ccdf, where='post', color=color)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(
        0.6, .95, '$P(K > k)$',
        fontsize=8,
        ha="center", va="top",
        transform=ax.transAxes)
    ax.text(
        0.55, .05, '$k$',
        fontsize=8,
        ha="center", va="bottom",
        transform=ax.transAxes)


In [ ]:
def plot_hist(
        degrees: NodeVector,
        minority: BinaryClassNodeVector,
        ax: plt.Axes
):
    bins = np.arange(1, max(degrees)+2)
    ax.hist(
        [degrees[minority==True], degrees[minority==False]],
        bins=bins,
        color=[COLOR_MIN, COLOR_MAJ],
        density=True,
        align='left',
        rwidth=0.8,
    )

In [ ]:
def plot_kde(
        degrees: NodeVector,
        minority: BinaryClassNodeVector,
        ax: plt.Axes
):
    import seaborn as sns
    sns.kdeplot(
        data=degrees[minority==True],
        fill=True,
        color=COLOR_MIN,
        alpha=0.5,
        bw_adjust=0.5,
        ax=ax,
        label='Minority'
    )
    sns.kdeplot(
        data=degrees[minority==False],
        fill=True,
        color=COLOR_MAJ,
        alpha=0.5,
        bw_adjust=0.5,
        ax=ax,
        label='Majority'
    )

In [ ]:
metrics = (
    "Segregation",
    "Degree inequality",
    "Inequity"
)

In [ ]:
fig, a_ax_nets = plt.subplots(ncols=3, figsize=((4/5)*SIZE_FIG[0], SIZE_FIG_SMALL[1]))

for ax, graph, pos, metric, config in zip(a_ax_nets, graphs, positions, metrics, model_configs):
    colors = [
        COLOR_MIN\
            if graph.nodes[n][CLASS_ATTRIBUTE] == MINORITY_VALUE else COLOR_MAJ\
                for n in graph.nodes]
    nx.draw(
        graph, pos,
        ax=ax,
        node_color=colors,
        with_labels=False,
        node_size=[
            nx.degree(graph, node)**(1.6) / 1.5 for node in graph.nodes
        ],
        edge_color="gray",
        width=0.5
        )

    ax.text(
        0, 0,
        s=(
            f"{MAP_LFM_SHORT[config.lfm_global.value]},{MAP_LFM_SHORT[config.lfm_tc.value]}\n"
            f"$h={config.homophily:.2f}$\n"
            f"$\\tau={config.tau:.2f}$"),
        ha="left",
        va="bottom",
        transform=ax.transAxes,
        fontsize=8,
        bbox=dict(
            facecolor='white',
            edgecolor='none',
            alpha=0.75,
            boxstyle='round,pad=0.15'
        )
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(
        0.5, .95,
        metric,
        fontsize=8,
        va="bottom",
        ha="center",
        transform=ax.transAxes)

ax_lorenz = a_ax_nets[1].inset_axes([.6, .0, .35, .35])
plot_lorenz_curve(degrees[1], minorities[1], ax_lorenz)

ax_ccdf = a_ax_nets[2].inset_axes([.6, .0, .35, .35])
plot_ccdf(degrees[2], minorities[2], ax_ccdf)
fig.savefig(f"../{PATH_PLOTS}/example_graphs.pdf")
